# Similitud semántica entre resúmenes de libros — "It" como pivote

Este notebook toma como referencia el código de la ficha de modelo de
[`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
(tokenización + mean pooling + normalización L2) y lo aplica al dataset de libros
(`titulos_dummy_1pct.csv`, `5pct`, `10pct`, `30pct`), usando **"It" de Stephen King**
como pivote: comparamos el embedding de su resumen contra el de todos los demás libros
del catálogo, usando las **6 formas de medir similitud/distancia coseno**.

> Nota: los 4 CSV son muestras del mismo catálogo a distinto porcentaje (no son
> particiones disjuntas necesariamente) — este notebook los une y deduplica por
> `articuloTitulo` antes de embeber, así el pivote y el resto del catálogo quedan
> en un único DataFrame.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import util
from scipy.spatial.distance import cosine as scipy_cosine

pd.set_option('display.max_colwidth', 90)


## 1. Carga y unificación del catálogo

Unimos las 4 muestras (`1%`, `5%`, `10%`, `30%`), guardamos de qué muestra viene
cada fila (columna `muestra`) y deduplicamos por título, porque el mismo libro
puede aparecer en más de un CSV.


In [ ]:
import pandas as pd

catalogo = pd.read_csv('titulos_dummy_50pct.csv')
catalogo = catalogo.drop_duplicates(
    subset="articuloTitulo"
).reset_index(drop=True)
catalogo = catalogo.drop_duplicates(subset="articuloTitulo").reset_index(drop=True)

print(f"Libros únicos en el catálogo: {len(catalogo)}")
catalogo

Libros únicos en el catálogo: 239


,articuloTitulo,resumen
0,Terror In Shadows : Someone Is Watching Me - L...,Novela de terror en inglés sobre una protagoni...
1,Libro: Cuentos Completos / Edgar Allan Poe,Recopilación de los cuentos completos de Edgar...
2,Pack Serie Completa Adabana 1 Al 3 - Non - Ivr...,Pack con los tres tomos completos del manga Ad...
3,"Frankenstein - Loqueleo Roja, de Wollstonecraf...","Edición de Frankenstein, de Mary Shelley, en l..."
4,"El Vampiro - Guillot, Victor Juan, De Guillot,...",Novela sobre un vampiro del autor Victor Juan ...
...,...,...
234,LA LLAMADA DEL CTHULHU Y OTROS CUENTOS DE TERR...,La llamada de Cthulhu y otros relatos de H. P....
235,El Valle Del Terror - Arthur Conan Doyle - Libro,"El valle del terror, de Arthur Conan Doyle: Sh..."
236,"Casa De Los Siete Tejados, La, De Hawthorne, N...","La casa de los siete tejados, de Nathaniel Haw..."
237,La Suerte Doblo La Esquina - Antologia - Quipu,Antología de relatos de la editorial Quipu.


## 2. Ubicar el pivote: "It" de Stephen King

Buscamos por título (empieza con "It") y confirmamos que el resumen menciona a
Stephen King, para no confundirlo con otro título que arranque con esas letras.


In [ ]:
mask_pivote = catalogo["articuloTitulo"].str.contains(
    r"\bIt\b.*Stephen King",
    case=False,
    regex=True,
    na=False
)

In [ ]:
mask_pivote.describe()

,articuloTitulo
count,239
unique,2
top,False
freq,238


In [ ]:
assert mask_pivote.sum() == 1, "Revisar: se esperaba encontrar exactamente un pivote."
idx_pivote = catalogo[mask_pivote].index[0]

print("Pivote encontrado en índice:", idx_pivote)
catalogo.loc[[idx_pivote]]


Pivote encontrado en índice: 182


,articuloTitulo,resumen
182,It Stephen King Español Debolsillo,"It, de Stephen King: un grupo de amigos se enf..."


## 3. Embeddings con `all-MiniLM-L6-v2` (tokenización + mean pooling + normalización)

Igual que en la ficha de modelo de Hugging Face: tokenizamos todos los resúmenes
en batch, pasamos por el modelo, aplicamos **mean pooling** sobre los tokens
(ponderado por la máscara de atención) y normalizamos con `F.normalize(p=2, dim=1)`.
Esa normalización es la que después habilita el "atajo" del punto 2 de la tabla.


In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # (batch, seq_len, hidden)
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model.eval()

resumenes = catalogo["resumen"].tolist()

encoded_input = tokenizer(
    resumenes, padding=True, truncation=True, return_tensors="pt"
)

with torch.no_grad():
    model_output = model(**encoded_input)

sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

print("Shape de los embeddings:", sentence_embeddings.shape)  # (n_libros, 384)


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape de los embeddings: torch.Size([239, 384])


## 4. Las 6 técnicas, "It" (pivote) contra cada libro del catálogo

Replicamos exactamente las 6 formas de la consigna. Para cada libro `j` del
catálogo comparamos `a = sentence_embeddings[idx_pivote]` contra
`b = sentence_embeddings[j]`.


In [ ]:
a_t = sentence_embeddings[idx_pivote]          # tensor, ya normalizado
a_np = a_t.numpy()

filas = []
for j in range(len(catalogo)):
    if j == idx_pivote:
        continue

    b_t = sentence_embeddings[j]
    b_np = b_t.numpy()

    # 1. Manual con fórmula completa
    cos_sim_manual = np.dot(a_np, b_np) / (
        np.linalg.norm(a_np) * np.linalg.norm(b_np)
    )

    # 2. Atajo (dot product puro) — válido porque a y b ya están normalizados
    cos_sim_atajo = np.dot(a_np, b_np)

    # 3. Torch (F.cosine_similarity) — requiere batch dimension
    cos_sim_torch = F.cosine_similarity(
        a_t.unsqueeze(0), b_t.unsqueeze(0)
    ).item()

    # 4. sentence-transformers (util.cos_sim)
    cos_sim_util = util.cos_sim(a_t, b_t).item()

    # 5. Distancia coseno manual
    cos_dist_manual = 1 - cos_sim_manual

    # 6. Distancia coseno con scipy (devuelve distancia directamente, float64)
    cos_dist_scipy = scipy_cosine(a_np, b_np)

    filas.append({
        "articuloTitulo": catalogo.loc[j, "articuloTitulo"],
    #    "muestra": catalogo.loc[j, "muestra"],
        "cos_sim_manual": cos_sim_manual,
        "cos_sim_atajo": cos_sim_atajo,
        "cos_sim_torch": cos_sim_torch,
        "cos_sim_util": cos_sim_util,
        "cos_dist_manual": cos_dist_manual,
        "cos_dist_scipy": cos_dist_scipy,
    })

resultados = pd.DataFrame(filas).sort_values("cos_sim_util", ascending=False).reset_index(drop=True)
resultados.head(10)


,articuloTitulo,cos_sim_manual,cos_sim_atajo,cos_sim_torch,cos_sim_util,cos_dist_manual,cos_dist_scipy
0,"Fairy Tale - A Novel - Stephen King, De King, Stephen. Editorial Scribner, Tapa Blanda...",0.671023,0.671023,0.671023,0.671023,0.328977,0.328977
1,Libro El Resplandor,0.662717,0.662717,0.662717,0.662717,0.337283,0.337283
2,"Libro El misterio de Salem's Lot (Ed. 50.º aniversario), de King, Stephen, vol. 1. Edi...",0.644391,0.644391,0.644391,0.644391,0.355609,0.355609
3,La Cupula | Stephen King,0.620450,0.620450,0.620450,0.620450,0.379550,0.379550
4,"Lobos Del Calla ( La Torre Oscura 5 ), De King, Stephen. Editorial Debolsillo, Tapa Bl...",0.620009,0.620009,0.620009,0.620009,0.379991,0.379991
5,"Viento Por La Cerradura, El",0.617683,0.617683,0.617683,0.617683,0.382317,0.382317
6,"Misery, de Stephen King. Editorial Debolsillo, tapa blanda en español, 2019",0.607059,0.607059,0.607059,0.607059,0.392941,0.392941
7,"Misery, De King, Stephen. Editorial Debolsillo, Tapa Blanda En Español, 2018",0.607059,0.607059,0.607059,0.607059,0.392941,0.392941
8,"La Milla Verde, De King, Stephen. Editorial Debolsillo, Tapa Blanda En Español",0.602538,0.602538,0.602538,0.602538,0.397462,0.397462
9,"Las Cuatro Estaciones II De King, Stephen Editorial Debolsillo Tapa Blanda En Español",0.601087,0.601087,0.601087,0.601087,0.398913,0.398913


## 5. Chequeo de consistencia entre técnicas

Las columnas de similitud (1, 2, 3, 4) deberían coincidir casi a la perfección
entre sí (float32), y `cos_dist_manual` debería ser exactamente `1 - cos_sim_manual`.
`cos_dist_scipy` puede diferir en el 5°-6° decimal respecto de `1 - cos_sim_util`
porque scipy castea a float64 — no es un error de cálculo, es precisión numérica.


In [ ]:
chequeo = resultados.iloc[0][[
    "cos_sim_manual", "cos_sim_atajo", "cos_sim_torch", "cos_sim_util",
    "cos_dist_manual", "cos_dist_scipy",
]]
print(chequeo)

diff_manual_vs_atajo = abs(resultados["cos_sim_manual"] - resultados["cos_sim_atajo"]).max()
diff_dist_manual_vs_scipy = abs(
    (1 - resultados["cos_sim_util"]) - resultados["cos_dist_scipy"]
).max()

print(f"\nMáxima diferencia manual vs. atajo (debería ser ~0): {diff_manual_vs_atajo:.2e}")
print(f"Máxima diferencia distancia (util) vs. scipy (float32 vs float64): {diff_dist_manual_vs_scipy:.2e}")


cos_sim_manual     0.671023
cos_sim_atajo      0.671023
cos_sim_torch      0.671023
cos_sim_util       0.671023
cos_dist_manual    0.328977
cos_dist_scipy     0.328977
Name: 0, dtype: object

Máxima diferencia manual vs. atajo (debería ser ~0): 5.96e-08
Máxima diferencia distancia (util) vs. scipy (float32 vs float64): 1.19e-07


## 6. Top-10 libros más parecidos a "It" por resumen

Ordenado por `cos_sim_util` (equivalente a cualquiera de las otras 3 columnas
de similitud, salvo ruido de precisión).


In [ ]:
resultados[[
    "articuloTitulo", "cos_sim_util", "cos_dist_scipy"
]].head(10)


,articuloTitulo,cos_sim_util,cos_dist_scipy
0,"Fairy Tale - A Novel - Stephen King, De King, Stephen. Editorial Scribner, Tapa Blanda...",0.671023,0.328977
1,Libro El Resplandor,0.662717,0.337283
2,"Libro El misterio de Salem's Lot (Ed. 50.º aniversario), de King, Stephen, vol. 1. Edi...",0.644391,0.355609
3,La Cupula | Stephen King,0.620450,0.379550
4,"Lobos Del Calla ( La Torre Oscura 5 ), De King, Stephen. Editorial Debolsillo, Tapa Bl...",0.620009,0.379991
5,"Viento Por La Cerradura, El",0.617683,0.382317
6,"Misery, de Stephen King. Editorial Debolsillo, tapa blanda en español, 2019",0.607059,0.392941
7,"Misery, De King, Stephen. Editorial Debolsillo, Tapa Blanda En Español, 2018",0.607059,0.392941
8,"La Milla Verde, De King, Stephen. Editorial Debolsillo, Tapa Blanda En Español",0.602538,0.397462
9,"Las Cuatro Estaciones II De King, Stephen Editorial Debolsillo Tapa Blanda En Español",0.601087,0.398913


## 7. Escalar a la matriz NxN completa con `util.cos_sim`

Esta es la fila que vale la pena remarcarles a los alumnos: de las 6 técnicas,
`util.cos_sim` es la única que escala directo a la matriz completa
(`util.cos_sim(emb, emb)`) sin reescribir el loop — lo mismo que necesitamos acá
para ubicar a "It" dentro de todo el catálogo (no solo comparado 1 a 1).


In [ ]:
matriz_sim = util.cos_sim(sentence_embeddings, sentence_embeddings)  # (n_libros, n_libros)
print("Shape de la matriz de similitud:", matriz_sim.shape)

fila_pivote = matriz_sim[idx_pivote]
top_k = torch.topk(fila_pivote, k=11)  # 11 porque el primero es "It" contra sí mismo (sim=1)

vecinos = pd.DataFrame({
    "articuloTitulo": catalogo.loc[top_k.indices.numpy(), "articuloTitulo"].values,
    #"muestra": catalogo.loc[top_k.indices.numpy(), "muestra"].values,
    "cos_sim": top_k.values.numpy(),
})
vecinos


Shape de la matriz de similitud: torch.Size([239, 239])


,articuloTitulo,cos_sim
0,It Stephen King Español Debolsillo,1.000000
1,"Fairy Tale - A Novel - Stephen King, De King, Stephen. Editorial Scribner, Tapa Blanda...",0.671023
2,Libro El Resplandor,0.662717
3,"Libro El misterio de Salem's Lot (Ed. 50.º aniversario), de King, Stephen, vol. 1. Edi...",0.644391
4,La Cupula | Stephen King,0.620450
5,"Lobos Del Calla ( La Torre Oscura 5 ), De King, Stephen. Editorial Debolsillo, Tapa Bl...",0.620009
6,"Viento Por La Cerradura, El",0.617683
7,"Misery, de Stephen King. Editorial Debolsillo, tapa blanda en español, 2019",0.607059
8,"Misery, De King, Stephen. Editorial Debolsillo, Tapa Blanda En Español, 2018",0.607059
9,"La Milla Verde, De King, Stephen. Editorial Debolsillo, Tapa Blanda En Español",0.602538


## Tabla resumen de las 6 técnicas

| Técnica | Qué devuelve | Requiere normas propias | Escala a matriz NxN sin cambios | Precisión interna |
|---|---|---|---|---|
| Manual (fórmula completa) | Similitud | Sí | No (hay que loopear) | float32 |
| Atajo (`np.dot`) | Similitud | No (asume normalizado) | No (hay que loopear) | float32 |
| Torch (`F.cosine_similarity`) | Similitud | No | No directo (par a par) | float32 |
| sentence-transformers (`util.cos_sim`) | Similitud | No | **Sí** (`util.cos_sim(emb, emb)`) | float32 |
| Distancia manual (`1 - sim`) | Distancia | Sí (hereda de manual) | No | float32 |
| scipy (`scipy_cosine`) | Distancia | No (la calcula internamente) | No (solo par a par) | float64 |


##Notas de REGEX

In [ ]:
titulo = "aaaa It Stephen King Español Debolsillo"

print("Título:", titulo)

print("\n1. Regex anterior: r'^It\\b'")
print(bool(pd.Series([titulo]).str.contains(
    r"^ It\b",
    case=False,
    regex=True
).iloc[0]))

print("\n2. Buscando 'It':")
print(bool(pd.Series([titulo]).str.contains(
    r"It",
    case=False,
    regex=True
).iloc[0]))

print("\n3. Buscando 'It' como palabra completa:")
print(bool(pd.Series([titulo]).str.contains(
    r"\bIt\b",
    case=False,
    regex=True
).iloc[0]))

print("\n4. Buscando 'It' + 'Stephen King':")
print(bool(pd.Series([titulo]).str.contains(
    r"It.*Stephen King",
    case=False,
    regex=True
).iloc[0]))

print("\n5. Regex final:")
print(bool(pd.Series([titulo]).str.contains(
    r"\bIt\b.*Stephen King",
    case=False,
    regex=True
).iloc[0]))

Título: It Stephen King Español Debolsillo

1. Regex anterior: r'^Dr\b'
False

2. Buscando 'It':
True

3. Buscando 'It' como palabra completa:
True

4. Buscando 'It' + 'Stephen King':
True

5. Regex final:
True
